In [1]:
import wrds
import pandas as pd
import numpy as np
from typing import Tuple, List

def g_cmp(d: wrds.Connection, t: str) -> pd.DataFrame:
    """Retrieves global fundamental data from Compustat.

    Args:
        d: Active WRDS connection object.
        t: Target ticker symbol.

    Returns:
        DataFrame containing historical fundamentals.
    """
    q = f"SELECT datadate, at, lt, nit, revt FROM comp.g_funda WHERE gvkey IN (SELECT gvkey FROM comp.g_security WHERE tic = '{t}') AND fic = 'AUS' AND indfmt = 'INDL' AND datafmt = 'STD' AND popsrc = 'I' AND consol = 'C' ORDER BY datadate ASC"
    return d.raw_sql(q, date_cols=['datadate'])

def g_ibs(d: wrds.Connection, t: str) -> pd.DataFrame:
    """Retrieves consensus estimates from LSEG IBES Summary.

    Args:
        d: Active WRDS connection object.
        t: Target ticker symbol.

    Returns:
        DataFrame containing EPS summary estimates.
    """
    q = f"SELECT statpers, fpedats, meanest, numest FROM ibes.statsumu_epsus WHERE ticker = '{t}' ORDER BY statpers ASC"
    return d.raw_sql(q, date_cols=['statpers', 'fpedats'])

def g_crs(d: wrds.Connection, p: int, s: str, e: str) -> pd.DataFrame:
    """Retrieves daily stock data from CRSP Version 2.

    Args:
        d: Active WRDS connection object.
        p: PERMNO identifier.
        s: Start date (YYYY-MM-DD).
        e: End date (YYYY-MM-DD).

    Returns:
        DataFrame containing daily returns and volumes.
    """
    q = f"SELECT dlycaldt, dlyret, dlyvol FROM crsp.dsf_v2 WHERE permno = {p} AND dlycaldt >= '{s}' AND dlycaldt <= '{e}' ORDER BY dlycaldt ASC"
    return d.raw_sql(q, date_cols=['dlycaldt'])

def g_evt(d: wrds.Connection, t: str) -> pd.DataFrame:
    """Retrieves key developments for event studies from Capital IQ.

    Args:
        d: Active WRDS connection object.
        t: Target ticker symbol.

    Returns:
        DataFrame containing event dates and descriptions.
    """
    q = f"SELECT a.announcedate, a.keydeveventtypeid, b.headline FROM ciq.wrds_keydev a JOIN ciq.ciqkeydev b ON a.keydevid = b.keydevid WHERE a.companyid IN (SELECT companyid FROM ciq.wrds_ticker WHERE ticker = '{t}') ORDER BY a.announcedate ASC"
    return d.raw_sql(q, date_cols=['announcedate'])

def g_ff(d: wrds.Connection, s: str, e: str) -> pd.DataFrame:
    """Retrieves Fama-French 5-Factor daily data.

    Args:
        d: Active WRDS connection object.
        s: Start date (YYYY-MM-DD).
        e: End date (YYYY-MM-DD).

    Returns:
        DataFrame containing daily risk factors.
    """
    q = f"SELECT date, mktrf, smb, hml, rmw, cma, rf FROM ff.fivefactors_daily WHERE date >= '{s}' AND date <= '{e}' ORDER BY date ASC"
    try:
        return d.raw_sql(q, date_cols=['date']).rename(columns={'date': 'ff_date'})
    except Exception:
        return pd.DataFrame(columns=['ff_date', 'mktrf', 'smb', 'hml', 'rmw', 'cma', 'rf'])

def g_esg(d: wrds.Connection, t: str) -> pd.DataFrame:
    """Retrieves ESG/Governance proxies dynamically.

    Args:
        d: Active WRDS connection object.
        t: Target ticker symbol.

    Returns:
        DataFrame containing historical ESG scores or empty schema if restricted.
    """
    q = f"SELECT meeting_date as as_of_date, female_directors, minority_directors FROM iss_directors_global.company_diversity WHERE ticker = '{t}' ORDER BY meeting_date ASC"
    try:
        return d.raw_sql(q, date_cols=['as_of_date'])
    except Exception:
        return pd.DataFrame(columns=['as_of_date', 'female_directors', 'minority_directors'])

def g_bdx(d: wrds.Connection, t: str) -> pd.DataFrame:
    """Retrieves BoardEx Director Network data for Rest of World entities.

    Args:
        d: Active WRDS connection object.
        t: Target ticker symbol.

    Returns:
        DataFrame containing annual board characteristics.
    """
    q = f"SELECT a.annual_report_date, AVG(b.network_size) as avg_board_network FROM boardex_row.row_wrds_org_summary a JOIN boardex_row.row_wrds_individual_networks b ON a.companyid = b.companyid WHERE a.ticker = '{t}' GROUP BY a.annual_report_date ORDER BY a.annual_report_date ASC"
    try:
        return d.raw_sql(q, date_cols=['annual_report_date'])
    except Exception:
        return pd.DataFrame(columns=['annual_report_date', 'avg_board_network'])

def b_pipe(c: pd.DataFrame, i: pd.DataFrame, r: pd.DataFrame, e: pd.DataFrame, ff: pd.DataFrame, esg: pd.DataFrame, bdx: pd.DataFrame) -> pd.DataFrame:
    """Builds an aligned, forward-filled feature matrix using asynchronous time-series merging.

    Args:
        c: Compustat fundamentals DataFrame.
        i: IBES estimates DataFrame.
        r: CRSP daily returns DataFrame.
        e: Capital IQ events DataFrame.
        ff: Fama-French factors DataFrame.
        esg: ISS ESG DataFrame.
        bdx: BoardEx DataFrame.

    Returns:
        DataFrame containing the fully integrated pipeline.
    """
    r = r.dropna(subset=['dlycaldt']).sort_values('dlycaldt')
    c = c.dropna(subset=['datadate']).sort_values('datadate')
    i = i.dropna(subset=['statpers']).sort_values('statpers')
    ff = ff.dropna(subset=['ff_date']).sort_values('ff_date')
    esg = esg.dropna(subset=['as_of_date']).sort_values('as_of_date')
    bdx = bdx.dropna(subset=['annual_report_date']).sort_values('annual_report_date')

    m = pd.merge_asof(r, c, left_on='dlycaldt', right_on='datadate', direction='backward')
    m = pd.merge_asof(m, i, left_on='dlycaldt', right_on='statpers', direction='backward')

    if not ff.empty:
        m = pd.merge_asof(m, ff, left_on='dlycaldt', right_on='ff_date', direction='backward')
    if not esg.empty:
        m = pd.merge_asof(m, esg, left_on='dlycaldt', right_on='as_of_date', direction='backward')
    if not bdx.empty:
        m = pd.merge_asof(m, bdx, left_on='dlycaldt', right_on='annual_report_date', direction='backward')

    e_g = e.groupby('announcedate').agg({'keydeveventtypeid': list, 'headline': list}).reset_index()
    f = pd.merge(m, e_g, left_on='dlycaldt', right_on='announcedate', how='left')

    d_cols = ['datadate', 'statpers', 'ff_date', 'as_of_date', 'annual_report_date', 'announcedate']
    f.drop(columns=[col for col in d_cols if col in f.columns], inplace=True)

    return f

In [2]:
def g_sch(d: wrds.Connection, k: str) -> List[str]:
    """Retrieves fully qualified table names matching a schema keyword.

    Args:
        d: Active WRDS connection object.
        k: Schema search keyword.

    Returns:
        List of fully qualified table strings.
    """
    q = f"SELECT table_schema || '.' || table_name AS f_tbl FROM information_schema.tables WHERE table_schema LIKE '%%{k}%%'"
    return d.raw_sql(q)['f_tbl'].tolist()

In [3]:
usr = "zackienzle1"
tk = "WDS"
p_no = 12345
st = "2014-01-01"
ed = "2024-01-01"

db = wrds.Connection(wrds_username=usr)

Loading library list...
Done


In [4]:
# iss_tbls = g_sch(db, 'iss')
# bdx_tbls = g_sch(db, 'boardex')

# db.close()

# print(iss_tbls)
# print(bdx_tbls)

In [5]:
df_cmp = g_cmp(db, tk)
df_ibs = g_ibs(db, tk)
df_crs = g_crs(db, p_no, st, ed)
df_evt = g_evt(db, tk)
df_ff = g_ff(db, st, ed)
df_esg = g_esg(db, tk)
df_bdx = g_bdx(db, tk)

db.close()

df_main = b_pipe(df_cmp, df_ibs, df_crs, df_evt, df_ff, df_esg, df_bdx)

In [6]:
df_cmp

,datadate,at,lt,nit,revt


In [7]:
df_ibs

,statpers,fpedats,meanest,numest


In [8]:
df_crs

,dlycaldt,dlyret,dlyvol
0,2014-01-02,-0.018685,4922700.0
1,2014-01-03,-0.000889,1528500.0
2,2014-01-06,-0.009402,3119500.0
3,2014-01-07,0.012441,2732900.0
4,2014-01-08,0.010008,3419200.0
...,...,...,...
2511,2023-12-22,-0.00394,1243162.0
2512,2023-12-26,0.006454,1154527.0
2513,2023-12-27,-0.002172,928378.0
2514,2023-12-28,-0.008085,795618.0


In [9]:
df_evt

,announcedate,keydeveventtypeid,headline
0,1994-01-03,95,Woodside Petroleum Ltd.(ASX:WPL) added to FTSE...
1,1996-07-01,95,Woodside Petroleum Ltd.(ASX:WPL) added to Russ...
2,2000-03-31,95,Woodside Petroleum Ltd.(ASX:WPL) added to S&P/...
3,2000-03-31,95,Woodside Petroleum Ltd.(ASX:WPL) added to S&P/...
4,2000-03-31,95,Woodside Petroleum Ltd.(ASX:WPL) added to S&P/...
...,...,...,...
2291,2026-03-17,101,Woodside Energy Appoints Elizabeth Westcott as...
2292,2026-03-17,16,Woodside Energy Appoints Elizabeth Westcott as...
2293,2026-03-18,16,Woodside Energy Group Ltd Appoints Mark Cutifa...
2294,2026-03-25,31,Woodside Energy Expands Operations with Beaumo...


In [10]:
df_main

,dlycaldt,dlyret,dlyvol,at,lt,nit,revt,fpedats,meanest,numest,mktrf,smb,hml,rmw,cma,rf,keydeveventtypeid,headline
0,2014-01-02,-0.018685,4922700.0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,-0.0088,-0.0025,0.0017,-0.0032,0.0011,0.0,NaN,NaN
1,2014-01-03,-0.000889,1528500.0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,0.0003,0.0041,0.0003,-0.0035,0.0015,0.0,NaN,NaN
2,2014-01-06,-0.009402,3119500.0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,-0.0034,-0.0054,0.003,-0.0031,0.0005,0.0,"[55, 55, 55]","[Woodside Petroleum Ltd. to Report Q3, 2014 Re..."
3,2014-01-07,0.012441,2732900.0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,0.0068,0.0032,-0.0038,-0.0008,-0.003,0.0,[26],[Woodside Petroleum Ltd. Revises Earnings Guid...
4,2014-01-08,0.010008,3419200.0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,0.0004,-0.0001,-0.001,-0.0047,-0.0007,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2511,2023-12-22,-0.00394,1243162.0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,0.002,0.0061,0.001,-0.0064,0.002,0.0002,NaN,NaN
2512,2023-12-26,0.006454,1154527.0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,0.0048,0.0082,0.0044,-0.0032,-0.0015,0.0002,NaN,NaN
2513,2023-12-27,-0.002172,928378.0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,0.0016,0.0016,0.0011,-0.0032,-0.0014,0.0002,NaN,NaN
2514,2023-12-28,-0.008085,795618.0,NaN,NaN,NaN,NaN,NaT,NaN,NaN,-0.0001,-0.0039,0.0003,-0.0031,0.0016,0.0002,NaN,NaN
